# ODL Quantitative Data & Topic Analysis

This notebook covers:
- **VADER Sentiment Analysis** — rule-based sentiment scoring
- **Descriptive Statistics** — quantitative summary of text corpora
- **Topic Modeling** — LDA and NMF topic discovery
- **Keyword Frequency Analysis** — TF-IDF and raw counts
- **Visualization** — sentiment distributions, word clouds, topic heatmaps

## 0. Install & Import Dependencies

In [ ]:
# Run once to install required packages
# !pip install vaderSentiment pandas numpy matplotlib seaborn wordcloud scikit-learn nltk gensim openpyxl
# import nltk; nltk.download('punkt'); nltk.download('stopwords'); nltk.download('vader_lexicon')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from wordcloud import WordCloud

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

analyzer = SentimentIntensityAnalyzer()
STOP_WORDS = set(stopwords.words('english'))

## 1. Load Data

In [ ]:
# ── Configure your data source ─────────────────────────────────────────────
DATA_PATH = 'data/odl_data.csv'   # CSV | .xlsx | .json — change as needed
TEXT_COL  = 'response_text'       # column containing the free-text to analyse
GROUP_COL = 'group'               # optional grouping column (set None to skip)
DATE_COL  = 'date'                # optional date column  (set None to skip)

# ── Load ───────────────────────────────────────────────────────────────────
if DATA_PATH.endswith('.xlsx'):
    df = pd.read_excel(DATA_PATH)
elif DATA_PATH.endswith('.json'):
    df = pd.read_json(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
df[TEXT_COL] = df[TEXT_COL].astype(str)

print(f'Loaded {len(df):,} rows')
df.head()

## 2. Descriptive / Quantitative Statistics

In [ ]:
df['char_count'] = df[TEXT_COL].str.len()
df['word_count'] = df[TEXT_COL].str.split().str.len()
df['sent_count'] = df[TEXT_COL].str.count(r'[.!?]+')

desc = df[['char_count', 'word_count', 'sent_count']].describe().T
display(desc.round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes,
                           ['char_count', 'word_count', 'sent_count'],
                           ['Character count', 'Word count', 'Sentence count']):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color='steelblue')
    ax.set_title(label)
plt.suptitle('Text Length Distributions', y=1.02, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. VADER Sentiment Analysis

In [ ]:
def vader_scores(text: str) -> dict:
    scores = analyzer.polarity_scores(text)
    scores['label'] = (
        'positive' if scores['compound'] >= 0.05
        else 'negative' if scores['compound'] <= -0.05
        else 'neutral'
    )
    return scores

vader_df = df[TEXT_COL].apply(vader_scores).apply(pd.Series)
df = pd.concat([df, vader_df], axis=1)

print('Sentiment distribution:')
print(df['label'].value_counts(normalize=True).mul(100).round(1).to_string())
df[['compound', 'neg', 'neu', 'pos']].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(df['compound'], bins=50, kde=True, ax=axes[0], color='mediumseagreen')
axes[0].axvline(0.05,  color='green',  linestyle='--', label='+0.05 (positive)')
axes[0].axvline(-0.05, color='tomato', linestyle='--', label='-0.05 (negative)')
axes[0].set_title('VADER Compound Score Distribution')
axes[0].legend()

palette = {'positive': 'mediumseagreen', 'neutral': 'steelblue', 'negative': 'tomato'}
order   = ['positive', 'neutral', 'negative']
counts  = df['label'].value_counts()
bars = axes[1].bar([l.capitalize() for l in order],
                   [counts.get(l, 0) for l in order],
                   color=[palette[l] for l in order])
axes[1].bar_label(bars, fmt='%d', padding=3)
axes[1].set_title('Sentiment Label Counts')
axes[1].set_ylabel('Count')

plt.suptitle('VADER Sentiment Overview', fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
if GROUP_COL and GROUP_COL in df.columns:
    grp_sentiment = (
        df.groupby(GROUP_COL)['compound']
          .agg(['mean', 'median', 'std', 'count'])
          .round(3)
          .sort_values('mean', ascending=False)
    )
    display(grp_sentiment)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=df, x=GROUP_COL, y='compound', ax=ax,
                order=grp_sentiment.index, palette='Set2')
    ax.axhline(0, color='grey', linestyle='--')
    ax.set_title(f'Compound Sentiment by {GROUP_COL}')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
    plt.tight_layout(); plt.show()
else:
    print('GROUP_COL not set — skipping grouped sentiment plot.')

In [ ]:
if DATE_COL and DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    ts = (
        df.set_index(DATE_COL)['compound']
          .resample('M').mean()
          .dropna()
    )
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts.index, ts.values, marker='o', color='steelblue')
    ax.axhline(0, color='grey', linestyle='--')
    ax.fill_between(ts.index, ts.values, 0,
                    where=ts.values >= 0, alpha=0.2, color='mediumseagreen')
    ax.fill_between(ts.index, ts.values, 0,
                    where=ts.values < 0,  alpha=0.2, color='tomato')
    ax.set_title('Monthly Average Sentiment Over Time')
    ax.set_ylabel('Mean Compound Score')
    plt.tight_layout(); plt.show()
else:
    print('DATE_COL not set — skipping time-series sentiment plot.')

## 4. Text Preprocessing

In [ ]:
EXTRA_STOP = set()  # add domain-specific stop words here
ALL_STOPS  = STOP_WORDS | EXTRA_STOP

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)         # URLs
    text = re.sub(r'[^a-z\s]', ' ', text)                # non-alpha
    tokens = [t for t in text.split() if t not in ALL_STOPS and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df[TEXT_COL].apply(clean_text)
df[['clean_text']].head(3)

## 5. Keyword / TF-IDF Analysis

In [ ]:
TOP_N = 30

tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_mat = tfidf_vec.fit_transform(df['clean_text'])

tfidf_means = pd.Series(
    tfidf_mat.mean(axis=0).A1,
    index=tfidf_vec.get_feature_names_out()
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
tfidf_means.head(TOP_N).sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_title(f'Top {TOP_N} Terms by Mean TF-IDF Score')
ax.set_xlabel('Mean TF-IDF')
plt.tight_layout(); plt.show()

In [ ]:
all_text = ' '.join(df['clean_text'])
wc = WordCloud(width=900, height=450, background_color='white',
               colormap='viridis', max_words=150).generate(all_text)
fig, ax = plt.subplots(figsize=(13, 6))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Word Cloud — Full Corpus', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'positive': 'Greens', 'neutral': 'Blues', 'negative': 'Reds'}
for ax, label in zip(axes, ['positive', 'neutral', 'negative']):
    subset = ' '.join(df.loc[df['label'] == label, 'clean_text'])
    if not subset.strip():
        ax.set_visible(False); continue
    wc = WordCloud(width=500, height=350, background_color='white',
                   colormap=colors[label], max_words=80).generate(subset)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(f'{label.capitalize()} responses', fontsize=12)
plt.suptitle('Word Clouds by Sentiment Label', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 6. Topic Modeling — LDA

In [ ]:
N_TOPICS    = 6    # adjust based on corpus size
N_TOP_WORDS = 10

count_vec = CountVectorizer(max_features=5000, min_df=2, ngram_range=(1, 1))
count_mat = count_vec.fit_transform(df['clean_text'])
vocab     = count_vec.get_feature_names_out()

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    learning_method='batch',
    max_iter=20,
)
lda.fit(count_mat)

doc_topics = lda.transform(count_mat)
df['dominant_topic'] = doc_topics.argmax(axis=1)
df['topic_weight']   = doc_topics.max(axis=1)
print('Topic distribution:')
print(df['dominant_topic'].value_counts().sort_index())

In [ ]:
def top_words_table(model, feature_names, n=10):
    rows = []
    for i, comp in enumerate(model.components_):
        words = [feature_names[j] for j in comp.argsort()[:-n-1:-1]]
        rows.append({'Topic': i, 'Top Words': ', '.join(words)})
    return pd.DataFrame(rows)

display(top_words_table(lda, vocab, N_TOP_WORDS))

In [ ]:
n_show  = 20
top_idx = lda.components_.sum(axis=0).argsort()[:-n_show-1:-1]
heat_data = pd.DataFrame(
    lda.components_[:, top_idx],
    columns=vocab[top_idx],
    index=[f'Topic {i}' for i in range(N_TOPICS)]
)
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heat_data, cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Top-word weights per LDA topic')
plt.tight_layout(); plt.show()

In [ ]:
ct = pd.crosstab(df['dominant_topic'], df['label'], normalize='index').round(3) * 100
ct.index = [f'Topic {i}' for i in ct.index]
display(ct)

ct[['negative', 'neutral', 'positive']].plot.bar(
    stacked=True, figsize=(10, 5),
    color=['tomato', 'steelblue', 'mediumseagreen']
)
plt.title('Sentiment Distribution per Topic (%)')
plt.ylabel('%'); plt.xticks(rotation=0); plt.legend(title='Sentiment')
plt.tight_layout(); plt.show()

## 7. Topic Modeling — NMF (Alternative)

In [ ]:
nmf = NMF(n_components=N_TOPICS, random_state=42, max_iter=400)
nmf.fit(tfidf_mat)

nmf_topics = top_words_table(nmf, tfidf_vec.get_feature_names_out(), N_TOP_WORDS)
nmf_topics.columns = ['Topic', 'NMF Top Words']
display(nmf_topics)

## 8. Export Results

In [ ]:
OUT_COLS = [TEXT_COL, 'char_count', 'word_count',
            'compound', 'pos', 'neu', 'neg', 'label',
            'dominant_topic', 'topic_weight']
if GROUP_COL and GROUP_COL in df.columns:
    OUT_COLS.insert(1, GROUP_COL)
if DATE_COL and DATE_COL in df.columns:
    OUT_COLS.insert(1, DATE_COL)

export_cols = [c for c in OUT_COLS if c in df.columns]
df[export_cols].to_csv('odl_analysis_results.csv', index=False)
print('Saved → odl_analysis_results.csv')

top_words_table(lda, vocab, N_TOP_WORDS).to_csv('lda_topics.csv', index=False)
print('Saved → lda_topics.csv')